## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Load Dataset

In [2]:
data = pd.read_csv("wdbc.data", header=None)
data.head()

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## Separate Features and Target

In [4]:
X = data.iloc[:, 2:].values
y = (data.iloc[:, 1] == "M").astype(int).values

print("Dataset Shape:", X.shape)

Dataset Shape: (569, 30)


## Split Dataset (80/20)

In [5]:
np.random.seed(42)

indices = np.random.permutation(len(X))
split = int(0.8 * len(X))

train = indices[:split]
test = indices[split:]

X_train, X_test = X[train], X[test]
y_train, y_test = y[train], y[test]

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 455
Testing Samples: 114


## Normalize Data

In [6]:
mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)

std[std == 0] = 1

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

## Implement Perceptron From Scratch

In [7]:
class Perceptron:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.lr = learning_rate
        self.epochs = epochs

    def activation(self, z):
        return 1 if z >= 0 else 0

    def fit(self, X, y):
        self.weights = np.zeros(X.shape[1])
        self.bias = 0

        for epoch in range(self.epochs):
            for i in np.random.permutation(len(X)):
                z = np.dot(X[i], self.weights) + self.bias
                prediction = self.activation(z)
                error = y[i] - prediction
                self.weights += self.lr * error * X[i]
                self.bias += self.lr * error

    def predict(self, X):
        predictions = []
        for sample in X:
            z = np.dot(sample, self.weights) + self.bias
            predictions.append(self.activation(z))
        return np.array(predictions)

## Train Model

In [8]:
model = Perceptron(learning_rate=0.01, epochs=1000)

model.fit(X_train, y_train)

print("Training Completed")

Training Completed


## Make Predictions

In [9]:
y_pred = model.predict(X_test)

print("Actual:", y_test[:10])
print("Predicted:", y_pred[:10])

Actual: [1 1 0 1 1 0 0 0 1 0]
Predicted: [1 1 0 1 1 0 0 0 1 0]


## Evaluate Model

In [10]:
accuracy = np.mean(y_test == y_pred) * 100

TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

precision = TP / (TP + FP) if TP + FP else 0
recall = TP / (TP + FN) if TP + FN else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

print("Accuracy:", round(accuracy, 2), "%")
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

Accuracy: 92.98 %
Precision: 0.9333
Recall: 0.8936
F1 Score: 0.913


## Confusion Matrix

In [11]:
print("             Predicted")
print("             Benign  Malignant")
print("Benign:       ", TN, FP)
print("Malignant:    ", FN, TP)

             Predicted
             Benign  Malignant
Benign:        64 3
Malignant:     5 42
